# AI Image Caption Generator Using BLIP
### Generative AI ISE Project
Run this notebook on Google Colab (with free GPU) to get a 100% free public URL.

In [ ]:
# 1. Install dependencies
!pip install -q transformers torch torchvision gradio gtts pillow
print("Dependencies installed successfully!")

In [ ]:
# 2. Load Salesforce BLIP and launch Gradio with public share URL
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from gtts import gTTS
import gradio as gr
import tempfile

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Salesforce/blip-image-captioning-base"
print(f"Loading BLIP on {device}...")
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id).to(device)
print("Model loaded successfully!")

def generate_caption(image, prefix_prompt):
    if image is None:
        return "Please upload an image.", None
    image = image.convert("RGB")
    if prefix_prompt and prefix_prompt.strip():
        inputs = processor(images=image, text=prefix_prompt.strip(), return_tensors="pt").to(device)
    else:
        inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, num_beams=5)
    caption = processor.decode(out[0], skip_special_tokens=True).strip()
    caption = caption[0].upper() + caption[1:] + "."
    tts = gTTS(text=caption, lang="en")
    temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tts.save(temp_audio.name)
    return caption, temp_audio.name

with gr.Blocks(title="AI Image Caption Generator") as demo:
    gr.Markdown("# 🖼️ AI Image Caption Generator Using BLIP")
    gr.Markdown("Generates captions and audio narration for accessibility.")
    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="Upload Image")
            prefix_in = gr.Textbox(label="Optional Prompt / Guidance", placeholder="e.g., a photo of")
            btn = gr.Button("Generate Caption", variant="primary")
        with gr.Column():
            cap_out = gr.Textbox(label="Generated Description", lines=3)
            aud_out = gr.Audio(label="Spoken Audio (Accessibility Narration)")
    btn.click(fn=generate_caption, inputs=[img_in, prefix_in], outputs=[cap_out, aud_out])

# Setting share=True gives you a free public https://xxxx.gradio.live link
demo.launch(share=True)
